In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import random
from skimage.filters import threshold_sauvola, threshold_niblack
from joblib import Parallel, delayed
import json
from datetime import datetime
import IPython.display as display
from PIL import Image
import io
import time

%matplotlib inline

In [2]:
df = pd.read_csv(r"C:\Users\PC-LAB1\img_processing_lab\day05\data\mnist\archive\mnist_train.csv")

In [3]:
import pandas as pd
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Load Data
# Update 'mnist_train.csv' with your actual training file path
train_df = pd.read_csv(r"C:\Users\PC-LAB1\img_processing_lab\day05\data\mnist\archive\mnist_train.csv")
test_df = pd.read_csv(r"C:\Users\PC-LAB1\img_processing_lab\day05\data\mnist\archive\mnist_test.csv")

# Separate features (pixels) and target label
X_train = train_df.drop('label', axis=1)
y_train = train_df['label']

X_test = test_df.drop('label', axis=1)
y_test = test_df['label']

# 2. Preprocess: Normalize pixel values from [0, 255] to [0.0, 1.0]
# Normalization is critical for both SVM margin optimization and MLP gradient convergence
X_train_norm = X_train / 255.0
X_test_norm = X_test / 255.0

# ==========================================
# MODEL 1: Support Vector Machine (SVM)
# ==========================================
print("Training Support Vector Machine (SVM)...")
svm_model = SVC(kernel='rbf', C=5.0, gamma='scale', random_state=42)
svm_model.fit(X_train_norm, y_train)

y_pred_svm = svm_model.predict(X_test_norm)
print(f"SVM Test Accuracy: {accuracy_score(y_test, y_pred_svm) * 100:.2f}%\n")
print("SVM Classification Report:")
print(classification_report(y_test, y_pred_svm))

# ==========================================
# MODEL 2: Multi-Layer Perceptron (MLP)
# ==========================================
print("\nTraining Multi-Layer Perceptron (MLP)...")
# 2 hidden layers with 128 and 64 neurons
mlp_model = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    max_iter=50,
    random_state=42
)
mlp_model.fit(X_train_norm, y_train)

y_pred_mlp = mlp_model.predict(X_test_norm)
print(f"MLP Test Accuracy: {accuracy_score(y_test, y_pred_mlp) * 100:.2f}%\n")
print("MLP Classification Report:")
print(classification_report(y_test, y_pred_mlp))

Training Support Vector Machine (SVM)...
SVM Test Accuracy: 98.41%

SVM Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.99       980
           1       0.99      0.99      0.99      1135
           2       0.98      0.98      0.98      1032
           3       0.98      0.99      0.98      1010
           4       0.98      0.98      0.98       982
           5       0.99      0.98      0.98       892
           6       0.99      0.99      0.99       958
           7       0.98      0.98      0.98      1028
           8       0.98      0.98      0.98       974
           9       0.98      0.97      0.98      1009

    accuracy                           0.98     10000
   macro avg       0.98      0.98      0.98     10000
weighted avg       0.98      0.98      0.98     10000


Training Multi-Layer Perceptron (MLP)...
MLP Test Accuracy: 98.14%

MLP Classification Report:
              precision    recall  f1-score   supp

In [5]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

# 1. Load Data
train_df = pd.read_csv(r"C:\Users\PC-LAB1\img_processing_lab\day05\data\mnist\archive\mnist_train.csv")
test_df = pd.read_csv(r"C:\Users\PC-LAB1\img_processing_lab\day05\data\mnist\archive\mnist_test.csv")

X_train_raw = train_df.drop('label', axis=1).values / 255.0
y_train = train_df['label'].values

X_test_raw = test_df.drop('label', axis=1).values / 255.0
y_test = test_df['label'].values

# ==========================================
# 2. FEATURE ENGINEERING FUNCTION
# ==========================================
def extract_handcrafted_features(X_raw):
    num_samples = X_raw.shape[0]
    images = X_raw.reshape(-1, 28, 28)
    
    # Feature 1: Row and Column Sums (Density Profiles) -> 56 features
    row_sums = images.sum(axis=2)  # Shape: (N, 28)
    col_sums = images.sum(axis=1)  # Shape: (N, 28)
    
    # Feature 2: Bounding Box Dimensions & Aspect Ratio -> 4 features
    bbox_features = []
    for img in images:
        rows, cols = np.where(img > 0.1) # Threshold active pixel regions
        if len(rows) > 0:
            h = rows.max() - rows.min() + 1
            w = cols.max() - cols.min() + 1
            aspect_ratio = w / float(h) if h > 0 else 0
            area = h * w
        else:
            h, w, aspect_ratio, area = 0, 0, 0, 0
        bbox_features.append([h, w, aspect_ratio, area])
        
    bbox_features = np.array(bbox_features)
    
    # Concatenate Handcrafted Features -> 60 features
    return np.hstack([row_sums, col_sums, bbox_features])

print("Extracting domain-specific structural features...")
X_train_handcrafted = extract_handcrafted_features(X_train_raw)
X_test_handcrafted = extract_handcrafted_features(X_test_raw)

# ==========================================
# 3. PCA FEATURE SPACE REDUCTION
# ==========================================
print("Applying PCA to raw pixel space (k=50)...")
pca = PCA(n_components=50, random_state=42)
X_train_pca = pca.fit_transform(X_train_raw)
X_test_pca = pca.transform(X_test_raw)

# Combine Raw Pixels + PCA Features + Handcrafted Structural Features
X_train_combined = np.hstack([X_train_raw, X_train_pca, X_train_handcrafted])
X_test_combined = np.hstack([X_test_raw, X_test_pca, X_test_handcrafted])

# Standardize combined features for stable model optimization
scaler = StandardScaler()
X_train_final = scaler.fit_transform(X_train_combined)
X_test_final = scaler.transform(X_test_combined)

# ==========================================
# 4. TRAIN EVALUATED MODELS
# ==========================================
# Optimized SVM
print("\nTraining Enhanced SVM with PCA + Structural Features...")
svm_model = SVC(kernel='rbf', C=10.0, gamma='scale', random_state=42)
svm_model.fit(X_train_final, y_train)

y_pred_svm = svm_model.predict(X_test_final)
print(f"Enhanced SVM Test Accuracy: {accuracy_score(y_test, y_pred_svm) * 100:.2f}%")

# Optimized MLP
print("\nTraining Enhanced MLP with PCA + Structural Features...")
mlp_model = MLPClassifier(
    hidden_layer_sizes=(256, 128),
    activation='relu',
    solver='adam',
    max_iter=100,
    random_state=42
)
mlp_model.fit(X_train_final, y_train)

y_pred_mlp = mlp_model.predict(X_test_final)
print(f"Enhanced MLP Test Accuracy: {accuracy_score(y_test, y_pred_mlp) * 100:.2f}%")

Extracting domain-specific structural features...
Applying PCA to raw pixel space (k=50)...

Training Enhanced SVM with PCA + Structural Features...
Enhanced SVM Test Accuracy: 97.47%

Training Enhanced MLP with PCA + Structural Features...
Enhanced MLP Test Accuracy: 98.07%


In [7]:
import pandas as pd
import numpy as np
import cv2
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

# 1. Load Data
train_df = pd.read_csv(r"C:\Users\PC-LAB1\img_processing_lab\day05\data\mnist\archive\mnist_train.csv")
test_df = pd.read_csv(r"C:\Users\PC-LAB1\img_processing_lab\day05\data\mnist\archive\mnist_test.csv")

# Drop label to keep features purely unsupervised
X_train_raw = train_df.drop('label', axis=1).values / 255.0
y_train = train_df['label'].values

X_test_raw = test_df.drop('label', axis=1).values / 255.0
y_test = test_df['label'].values

# ==============================================================================
# STEP 1: FULL FEATURE EXTRACTION (Raw Pixels + Density + Bbox + Hu Moments)
# ==============================================================================
def extract_full_feature_space(X_raw):
    images = X_raw.reshape(-1, 28, 28)
    full_features = []
    
    for i, img in enumerate(images):
        # A. Raw Pixel Data (784 features)
        raw_pixels = X_raw[i]
        
        # B. Spatial Density Profiles (28 horizontal + 28 vertical = 56 features)
        row_sums = img.sum(axis=1)
        col_sums = img.sum(axis=0)
        
        # C. Bounding Box & Geometric Features (4 features)
        rows, cols = np.where(img > 0.1)
        if len(rows) > 0:
            h = rows.max() - rows.min() + 1
            w = cols.max() - cols.min() + 1
            aspect_ratio = w / float(h) if h > 0 else 0
            area = h * w
        else:
            h, w, aspect_ratio, area = 0, 0, 0, 0
            
        # D. Hu Invariant Moments (7 features)
        img_uint8 = (img * 255).astype(np.uint8)
        moments = cv2.moments(img_uint8)
        hu_raw = cv2.HuMoments(moments).flatten()
        
        # Log-transform Hu moments for stability
        hu_log = []
        for hu in hu_raw:
            if hu != 0:
                hu_log.append(-1.0 * np.copysign(1.0, hu) * np.log10(abs(hu)))
            else:
                hu_log.append(0.0)
                
        # Concatenate into one unified feature vector (851 dimensions total)
        combined_vector = np.hstack([
            raw_pixels, 
            row_sums, 
            col_sums, 
            [h, w, aspect_ratio, area], 
            hu_log
        ])
        full_features.append(combined_vector)
        
    return np.array(full_features)

print("Calculating full feature space for train and test sets...")
X_train_full = extract_full_feature_space(X_train_raw)
X_test_full = extract_full_feature_space(X_test_raw)

print(f"Full Feature Vector Shape: {X_train_full.shape}") 
# 784 (pixels) + 56 (densities) + 4 (bbox) + 7 (Hu moments) = 851 features

# ==============================================================================
# STEP 2: STANDARDIZATION BEFORE PCA
# ==============================================================================
# Essential because Hu moments, bounding box areas, and pixels have different units
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_full)
X_test_scaled = scaler.transform(X_test_full)

# ==============================================================================
# STEP 3: APPLY PCA ON THE COMPLETE FEATURE SPACE
# ==============================================================================
print("\nApplying PCA to reduce the 851-dimensional feature space...")
# Option A: Specify variance ratio (e.g., retain 95% of total variance)
pca = PCA(n_components=0.95, random_state=42)

# Option B: Specify fixed number of components (e.g., n_components=100)
# pca = PCA(n_components=100, random_state=42)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"PCA reduced feature dimensions from {X_train_full.shape[1]} down to {X_train_pca.shape[1]}")

# ==============================================================================
# STEP 4: MODEL TRAINING & EVALUATION
# ==============================================================================

# Model 1: Support Vector Machine (SVM)
print("\nTraining SVM on PCA-transformed feature representation...")
svm_model = SVC(kernel='rbf', C=10.0, gamma='scale', random_state=42)
svm_model.fit(X_train_pca, y_train)

y_pred_svm = svm_model.predict(X_test_pca)
print(f"SVM Test Accuracy: {accuracy_score(y_test, y_pred_svm) * 100:.2f}%\n")
print(classification_report(y_test, y_pred_svm))

# Model 2: Multi-Layer Perceptron (MLP)
print("\nTraining MLP on PCA-transformed feature representation...")
mlp_model = MLPClassifier(
    hidden_layer_sizes=(256, 128),
    activation='relu',
    solver='adam',
    max_iter=100,
    random_state=42
)
mlp_model.fit(X_train_pca, y_train)

y_pred_mlp = mlp_model.predict(X_test_pca)
print(f"MLP Test Accuracy: {accuracy_score(y_test, y_pred_mlp) * 100:.2f}%\n")
print(classification_report(y_test, y_pred_mlp))

Calculating full feature space for train and test sets...
Full Feature Vector Shape: (60000, 851)

Applying PCA to reduce the 851-dimensional feature space...
PCA reduced feature dimensions from 851 down to 327

Training SVM on PCA-transformed feature representation...
SVM Test Accuracy: 97.50%

              precision    recall  f1-score   support

           0       0.98      0.99      0.99       980
           1       0.99      0.99      0.99      1135
           2       0.97      0.97      0.97      1032
           3       0.98      0.98      0.98      1010
           4       0.98      0.97      0.98       982
           5       0.97      0.97      0.97       892
           6       0.99      0.98      0.98       958
           7       0.94      0.98      0.96      1028
           8       0.98      0.96      0.97       974
           9       0.98      0.96      0.97      1009

    accuracy                           0.97     10000
   macro avg       0.98      0.97      0.97     10000

In [11]:
import time
import numpy as np
import pandas as pd
import cv2

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# 1. Dataset Initialization
digits = load_digits()
X_all = digits.data / 16.0
y_all = digits.target

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.25, random_state=42, stratify=y_all
)

# ---------------------------------------------------------
# PIPELINE 1: Raw Pixel Support Vector Machine (RBF Kernel)
# ---------------------------------------------------------
t0 = time.time()
svm_raw = SVC(kernel='rbf', C=5.0, gamma='scale', random_state=42)
svm_raw.fit(X_train_raw, y_train)
svm_train_time = time.time() - t0

t0 = time.time()
y_pred_svm = svm_raw.predict(X_test_raw)
svm_infer_time = time.time() - t0

svm_acc = accuracy_score(y_test, y_pred_svm)
svm_p, svm_r, svm_f1, _ = precision_recall_fscore_support(y_test, y_pred_svm, average='macro')


# ---------------------------------------------------------
# PIPELINE 2: Feature Extraction -> PCA -> MLP Classifier
# ---------------------------------------------------------
def extract_full_features(X_data):
    images = X_data.reshape(-1, 8, 8)
    features_list = []
    
    for i, img in enumerate(images):
        raw_pixels = X_data[i]
        row_sums = img.sum(axis=1)
        col_sums = img.sum(axis=0)
        
        rows, cols = np.where(img > 0.1)
        if len(rows) > 0:
            h = rows.max() - rows.min() + 1
            w = cols.max() - cols.min() + 1
            aspect_ratio = w / float(h) if h > 0 else 0
            area = h * w
        else:
            h, w, aspect_ratio, area = 0, 0, 0, 0
            
        img_uint8 = (img * 255).astype(np.uint8)
        moments = cv2.moments(img_uint8)
        hu = cv2.HuMoments(moments).flatten()
        
        hu_log = []
        for h_val in hu:
            if h_val != 0:
                hu_log.append(-1.0 * np.copysign(1.0, h_val) * np.log10(abs(h_val)))
            else:
                hu_log.append(0.0)
                
        feat_vec = np.hstack([raw_pixels, row_sums, col_sums, [h, w, aspect_ratio, area], hu_log])
        features_list.append(feat_vec)
        
    return np.array(features_list)

t0 = time.time()
X_train_full = extract_full_features(X_train_raw)
X_test_full = extract_full_features(X_test_raw)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_full)
X_test_scaled = scaler.transform(X_test_full)

pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)
feat_prep_time = time.time() - t0

mlp_model = MLPClassifier(hidden_layer_sizes=(256, 128), activation='relu', solver='adam', max_iter=100, random_state=42)

t0 = time.time()
mlp_model.fit(X_train_pca, y_train)
mlp_train_time = time.time() - t0

t0 = time.time()
y_pred_mlp = mlp_model.predict(X_test_pca)
mlp_infer_time = time.time() - t0

mlp_acc = accuracy_score(y_test, y_pred_mlp)
mlp_p, mlp_r, mlp_f1, _ = precision_recall_fscore_support(y_test, y_pred_mlp, average='macro')


# ---------------------------------------------------------
# BENCHMARK REPORT
# ---------------------------------------------------------
benchmark_df = pd.DataFrame({
    'Metric': [
        'Accuracy', 
        'Precision (Macro)', 
        'Recall (Macro)', 
        'F1-Score (Macro)', 
        'Prep Time (s)', 
        'Train Time (s)', 
        'Inference Time (s)'
    ],
    'Pipeline 1 (Raw SVM)': [
        f"{svm_acc:.4f}", 
        f"{svm_p:.4f}", 
        f"{svm_r:.4f}", 
        f"{svm_f1:.4f}", 
        "0.0000", 
        f"{svm_train_time:.4f}", 
        f"{svm_infer_time:.4f}"
    ],
    'Pipeline 2 (Full Features + PCA + MLP)': [
        f"{mlp_acc:.4f}", 
        f"{mlp_p:.4f}", 
        f"{mlp_r:.4f}", 
        f"{mlp_f1:.4f}", 
        f"{feat_prep_time:.4f}", 
        f"{mlp_train_time:.4f}", 
        f"{mlp_infer_time:.4f}"
    ]
})

print("\n================ FINAL BENCHMARK SUMMARY ================")
print(benchmark_df.to_string(index=False))
print("=========================================================\n")


================ FINAL BENCHMARK SUMMARY ================
            Metric Pipeline 1 (Raw SVM) Pipeline 2 (Full Features + PCA + MLP)
          Accuracy               0.9911                                 0.9800
 Precision (Macro)               0.9914                                 0.9800
    Recall (Macro)               0.9910                                 0.9797
  F1-Score (Macro)               0.9911                                 0.9798
     Prep Time (s)               0.0000                                 0.0480
    Train Time (s)               0.0161                                 1.2402
Inference Time (s)               0.0130                                 0.0020

